# TUTORIAL <span style="color:orange;">***MASAÏ***</span><img width='64' style="float:right;margin:0 10px 0 20px;" src="images/masai.png" />

Back to the main [Index](Index.ipynb)

## Part 8. 2D EXSY processing

#### Various imports and graphics settings

In [1]:
from __future__ import print_function, division

#### Loading the <span style="color:orange;">***MASAÏ***</span> library

In [2]:
from masai.api import *

MASAI api loaded: version 0.2.4.32

The current directory is </Users/christian/Dropbox/D.PROGRAMMES/masai/doc/tutorial>
MASAI root directory is </Users/christian/Dropbox/D.PROGRAMMES/masai>


ImportError: cannot import name loc2index

#### Set up the general location of data

In [ ]:
import os
# You should set the MASAI environnement variable before running these exemples')
# or change the two following paths according to your installation
DATADIR = os.path.join(MASAI,'doc/tutorial/exemples')
FIGDIR = os.path.join(MASAI,'doc/tutorial/figures')

## Load and plot a 2D fid

In [ ]:
user = 'user'
name = 'EXSY'
expno = 451
path = os.path.join(DATADIR, user,'nmr',name, str(expno))
source = Bruker(path) 
source

In [ ]:
d8 = float(source.par.D[8])*1000.
print("D8 is the mixing time: %.2f ms"%d8)

## Process SER data

#### Process the first dimension

In [ ]:
source = Bruker(path) 

# Make a broadening shifted to the TOP of echoes 
# **REMARKS:** This is possible in ``STATES`` mode only (TO CHECK!)

Process(source, transform = 'em 2 --k_shifted 1 0')

# Zerofilling ( zf n : means n times TD)
Process(source, transform = 'zf 0') 

# FT
Process(source, transform = 'ft')

# clip the region of interest
Process(source, transform = 'cl -10 120')

In [ ]:
# Manually phasing and plot of the first row to check the phase
Process(source, transform = 'pk -15 0 --pivot 82')
row = source.get_row(0, byindex=True)
p_1 = Plot1d(row, reverse=True)

In [ ]:
proc = Process(source, transform = 'pk --interactive --pivot 82')

In [ ]:
# Apply the above interactive phase 
proc.apply()

In [ ]:
Plot2d(source, figsize=(3,2.7), ylim=(0,4.2))
row = source.get_row(0, byindex=True)
_ = Plot1d(row, reverse=True)

#### Process the second dimension

In [ ]:
Process(source, axis=0, transform='em 10')
Process(source, axis=0, transform='zf 0') 
Process(source, axis=0, transform='ft')

In [ ]:
Process(source, axis=0, transform='pk -65 -180 --pivot -0.4') #manual phasinf first needed
row = source.get_col(80.88, width=0.)
p_1 = Plot1d(row, reverse=True)

In [ ]:
proc2 = Process(source, axis=0, transform = 'pk --interactive -.7 81 97 --pivot -0.4')

In [ ]:
# Apply the above interactive phase 
proc2.apply()

In [ ]:
# display 2d
# clip the region of interest
Process(source, axis=0, transform = 'cl -10 120')

Plot2d(source, figsize=(3,2.7), exponent=1.5, nc=50)
row = source.get_col(81)
_ = Plot1d(row, reverse=True)

In [ ]:
# symetrisation

data = source.data.copy()
index= np.linspace(max(min(data.index), min(data.columns)),min(max(data.index), max(data.columns)),1024)
data = data.reindex(index = index, columns = index, method='nearest')
backup = source.copy()
print(data.shape)
backup.data = data + data.T
Plot2d(backup)